# N5 — Model Context Protocol (MCP)

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a> 

Este notebook introduzir o **Model Context Protocol**, ou **MCP**, como um padrão para conectar aplicações de IA a ferramentas, dados e workflows externos.

Nas aulas anteriores, trabalhamos com:

- fundamentos conceituais de agentes;
- grafos de execução com LangGraph;
- chamadas de LLM dentro de grafos;
- tools e tool calling.

A ideia central é entender o MCP como uma camada de padronização entre:

- agentes;
- ferramentas;
- fontes de dados;
- prompts reutilizáveis;
- aplicações externas.

Em vez de cada agente implementar uma integração própria para cada sistema, o MCP propõe um contrato comum para descoberta e uso dessas capacidades.


## 0.1 Pré-requisitos

Para acompanhar este notebook, é esperado que você já tenha:

- entendido o papel de tools em agentes baseados em LLMs;
- visto o padrão ReAct em alto nível;
- noções de mensagens estruturadas, como `AIMessage`, `ToolMessage` ou objetos JSON;
- familiaridade básica com dicionários e funções em Python;
- uma noção geral de APIs, mesmo que ainda não tenha implementado APIs reais.

Não é necessário ter um servidor MCP real instalado para executar os exemplos principais deste notebook.

Vamos implementar uma **simulação didática** do protocolo usando Python puro.


## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Explicar o problema de integração que o MCP tenta resolver.
2. Diferenciar **host**, **cliente MCP** e **servidor MCP**.
3. Entender o papel do protocolo base, das mensagens JSON-RPC e da negociação de capacidades.
4. Diferenciar os principais recursos expostos por servidores MCP: **resources**, **prompts** e **tools**.
5. Simular chamadas MCP de listagem e execução de tools.
6. Entender como o MCP se encaixa em arquiteturas com agentes e LangGraph.
7. Identificar cuidados de segurança, autorização e aprovação humana.
8. Comparar integrações ad hoc, tools locais e servidores MCP.


## 0.3 Mapa do notebook

Neste notebook, vamos passar por:

1. O problema: muitas integrações diferentes para muitos agentes.
2. O modelo mental do MCP.
3. Host, cliente e servidor.
4. Protocolo base e mensagens JSON-RPC.
5. Ciclo de inicialização e capacidades.
6. Resources: dados e contexto.
7. Prompts: templates e workflows reutilizáveis.
8. Tools: ações invocáveis pelo modelo.
9. Um mini-servidor MCP didático em Python.
10. Um mini-cliente MCP didático.
11. Uso do MCP dentro de um agente tutor.
12. Integração com múltiplos servidores.
13. Relação com LangChain, LangGraph e ReAct.
14. Boas práticas de segurança.
15. Exercícios de fixação.


## 0.4 Contexto usado nos exemplos

Para manter continuidade com os notebooks anteriores, vamos usar o mesmo cenário geral:

- existe um **curso sobre agentes com LLMs e LangGraph**;
- os módulos anteriores abordaram fundamentos, LangGraph, LLMs e tools;
- agora queremos expor informações do curso de maneira padronizada para agentes.

Nos exemplos, vamos criar um servidor MCP fictício chamado **CursoMCPServer**.

Ele irá expor:

- um recurso com a ementa do curso;
- um recurso com o cronograma;
- um prompt reutilizável para tutoria;
- tools para consultar módulos, sugerir materiais e calcular plano de estudo.

Tudo será simulado localmente, sem internet e sem servidor real.


---

## 0.5 Preparação do ambiente

Os exemplos principais usam apenas Python básico.

A célula abaixo importa utilitários usados ao longo da aula.

> Observação: ao final do notebook há uma seção opcional mostrando como a ideia se aproximaria de um servidor MCP real. Essa parte fica como referência e pode depender do SDK oficial instalado no ambiente.


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pprint import pprint
from typing import Any, Callable, Dict, List, Optional


Vamos criar uma função auxiliar para imprimir objetos JSON de forma legível.


In [ ]:
def print_json(value: Any) -> None:
    """Imprime objetos Python como JSON formatado."""
    print(json.dumps(value, ensure_ascii=False, indent=2))


# 1. O problema: integração ponto a ponto

Antes do MCP, uma aplicação de agente normalmente precisava saber detalhes específicos de cada integração.

Por exemplo, imagine que temos:

- três agentes diferentes;
- quatro fontes de dados;
- cinco ferramentas externas.

Sem um padrão comum, cada agente pode precisar de código próprio para cada sistema.

Isso cria um problema de integração **N × M**:

- cada nova aplicação precisa aprender como conversar com cada sistema;
- cada novo sistema precisa fornecer adaptadores específicos para várias aplicações;
- a lógica de autenticação, schemas, erros e descoberta tende a ficar espalhada.

O MCP tenta reduzir esse acoplamento.

A ideia é parecida com isto:

```text
sem MCP:

Agente A ---- integração própria ---- Banco X
Agente A ---- integração própria ---- API Y
Agente B ---- integração própria ---- Banco X
Agente B ---- integração própria ---- API Y

com MCP:

Agente / Host ---- Cliente MCP ---- Servidor MCP ---- Banco X, API Y, arquivos, workflows
```


Vamos representar o problema usando Python.


In [ ]:
agentes = ["TutorLangGraph", "AssistenteDeProjeto", "AgenteDeRevisao"]
sistemas_externos = ["cronograma", "materiais", "progresso_aluno", "base_de_exercicios"]

integracoes_ponto_a_ponto = [
    (agente, sistema)
    for agente in agentes
    for sistema in sistemas_externos
]

print("Quantidade de integrações necessárias sem um padrão comum:", len(integracoes_ponto_a_ponto))
pprint(integracoes_ponto_a_ponto)


Com um contrato comum, cada sistema pode expor suas capacidades por um servidor MCP.

Cada aplicação passa a precisar de um cliente capaz de falar MCP, em vez de uma integração customizada para cada sistema.


In [ ]:
servidores_mcp = ["CursoMCPServer", "AlunoMCPServer", "ExerciciosMCPServer"]

integracoes_com_mcp = [
    (agente, "Cliente MCP")
    for agente in agentes
] + [
    ("Cliente MCP", servidor)
    for servidor in servidores_mcp
]

print("Quantidade de conexões conceituais com MCP:", len(integracoes_com_mcp))
pprint(integracoes_com_mcp)


A comparação acima é simplificada, mas ajuda a fixar o ponto principal:

> MCP não é apenas uma forma diferente de chamar função. Ele é um contrato para descoberta, descrição, autorização e uso de capacidades externas por aplicações de IA.


# 2. Modelo mental do MCP

O MCP organiza a comunicação em três papéis principais.

## 2.1 Host

O **host** é a aplicação de IA que o usuário enxerga.

Exemplos conceituais:

- um chatbot;
- uma IDE com assistente de código;
- um agente em LangGraph;
- uma aplicação corporativa com LLM;
- um ambiente de automação.

O host gerencia a experiência do usuário, conversa com o modelo e decide quais clientes MCP serão usados.

## 2.2 Cliente MCP

O **cliente MCP** fica dentro do host.

Ele é o componente que fala o protocolo MCP com um servidor específico.

Um host pode ter vários clientes MCP ativos ao mesmo tempo, por exemplo:

- um cliente conectado ao servidor de arquivos;
- um cliente conectado ao servidor do calendário;
- um cliente conectado ao servidor do curso;
- um cliente conectado ao servidor de banco de dados.

## 2.3 Servidor MCP

O **servidor MCP** expõe capacidades externas de forma padronizada.

Ele pode expor:

- dados;
- arquivos;
- APIs;
- bancos de dados;
- prompts;
- ferramentas;
- workflows.


Uma forma simples de visualizar:

```text
Usuário
  |
  v
Host de IA
  |
  | possui um ou mais
  v
Clientes MCP
  |
  | falam JSON-RPC pelo transporte configurado
  v
Servidores MCP
  |
  | adaptam sistemas externos
  v
Dados, tools, APIs, arquivos, prompts e workflows
```


Vamos criar uma representação conceitual desses papéis.


In [ ]:
@dataclass
class HostIA:
    nome: str
    clientes_mcp: List[str] = field(default_factory=list)


@dataclass
class ClienteMCPConceitual:
    nome: str
    servidor_conectado: str
    transporte: str


@dataclass
class ServidorMCPConceitual:
    nome: str
    capacidades: List[str]


host = HostIA(
    nome="TutorLangGraph Host",
    clientes_mcp=["cliente_curso", "cliente_aluno"],
)

cliente_curso = ClienteMCPConceitual(
    nome="cliente_curso",
    servidor_conectado="CursoMCPServer",
    transporte="stdio",
)

servidor_curso = ServidorMCPConceitual(
    nome="CursoMCPServer",
    capacidades=["resources", "prompts", "tools"],
)

pprint(host)
pprint(cliente_curso)
pprint(servidor_curso)


# 3. MCP não substitui o agente

Um ponto importante:

> MCP não é o agente.

O MCP define como o agente pode descobrir e usar capacidades externas.

A lógica de agente ainda pode estar em:

- um loop ReAct manual;
- uma aplicação LangChain;
- um grafo LangGraph;
- uma aplicação própria;
- um orquestrador corporativo.

O MCP entra como camada de integração.

No notebook anterior, uma tool era uma função Python local decorada com `@tool`.

Com MCP, a tool pode estar em outro processo, outro serviço ou outra máquina, desde que o servidor exponha essa capacidade seguindo o protocolo.


Comparação conceitual:

| Conceito | Papel |
|---|---|
| LLM | Gera texto, interpreta contexto e decide ações prováveis |
| Agente | Controla ciclo de decisão, ação, observação e resposta |
| Tool local | Função exposta diretamente pela aplicação |
| Servidor MCP | Serviço que expõe tools, dados e prompts por um protocolo comum |
| Cliente MCP | Conector usado pelo host para falar com um servidor MCP |
| LangGraph | Orquestra estados, nós, arestas e ciclos do agente |


# 4. Protocolo base: mensagens JSON-RPC

O MCP usa mensagens JSON-RPC 2.0.

Em termos práticos, isso significa que cliente e servidor trocam objetos JSON com campos como:

- `jsonrpc`;
- `id`;
- `method`;
- `params`;
- `result`;
- `error`.

Existem três tipos principais de mensagem:

1. **Request**: pede que o outro lado execute uma operação.
2. **Response**: responde a uma request, com resultado ou erro.
3. **Notification**: envia um aviso sem esperar resposta.

Vamos criar funções auxiliares para montar essas mensagens.


In [ ]:
def jsonrpc_request(request_id: int | str, method: str, params: Optional[dict] = None) -> dict:
    message = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
    }

    if params is not None:
        message["params"] = params

    return message


def jsonrpc_result(request_id: int | str, result: dict) -> dict:
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "result": result,
    }


def jsonrpc_error(request_id: int | str | None, code: int, message: str, data: Any = None) -> dict:
    error = {
        "code": code,
        "message": message,
    }

    if data is not None:
        error["data"] = data

    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "error": error,
    }


def jsonrpc_notification(method: str, params: Optional[dict] = None) -> dict:
    message = {
        "jsonrpc": "2.0",
        "method": method,
    }

    if params is not None:
        message["params"] = params

    return message


Exemplo de request para inicializar uma conexão.


In [ ]:
initialize_message = jsonrpc_request(
    request_id=1,
    method="initialize",
    params={
        "protocolVersion": "2025-11-25",
        "clientInfo": {
            "name": "TutorLangGraphHost",
            "version": "0.1.0",
        },
        "capabilities": {},
    },
)

print_json(initialize_message)


Exemplo de response para a inicialização.


In [ ]:
initialize_response = jsonrpc_result(
    request_id=1,
    result={
        "protocolVersion": "2025-11-25",
        "serverInfo": {
            "name": "CursoMCPServer",
            "version": "0.1.0",
        },
        "capabilities": {
            "resources": {},
            "prompts": {},
            "tools": {"listChanged": False},
        },
    },
)

print_json(initialize_response)


Exemplo de notification.

Uma notification não possui `id`, porque não espera resposta.


In [ ]:
notification = jsonrpc_notification(
    method="notifications/initialized",
    params={"message": "cliente pronto para usar o servidor"},
)

print_json(notification)


# 5. Ciclo de inicialização e negociação de capacidades

Antes de usar um servidor MCP, o cliente precisa descobrir informações básicas:

- qual versão do protocolo será usada;
- qual é o nome e versão do servidor;
- quais capacidades o servidor oferece;
- quais capacidades o cliente oferece.

Esse momento é importante porque o servidor pode expor apenas parte do protocolo.

Um servidor simples pode expor apenas `tools`.

Outro servidor pode expor `resources`, `prompts` e `tools`.

Outro pode não executar nenhuma ação sensível, apenas fornecer contexto.


Vamos criar dados simulados do nosso curso.


In [ ]:
CRONOGRAMA_CURSO = {
    "A1-A3": {
        "titulo": "Fundamentos Conceituais de Agentes com LLMs",
        "foco": "conceitos de agentes, autonomia, prompts, estado e especificação",
        "tipo": "conceitual",
    },
    "A4": {
        "titulo": "Fundamentos Práticos de LangGraph",
        "foco": "StateGraph, estado, nós, arestas, reducers e ciclos",
        "tipo": "prático",
    },
    "A5": {
        "titulo": "LangGraph com LLMs",
        "foco": "mensagens, LLMs, roteamento, revisão e ciclos com modelo",
        "tipo": "prático",
    },
    "A6": {
        "titulo": "Tools e agentes ReAct",
        "foco": "tools, ToolMessage, tool calling e loop ReAct",
        "tipo": "prático",
    },
    "A7": {
        "titulo": "Model Context Protocol para agentes, tools e dados",
        "foco": "padronização de integrações entre agentes, ferramentas e fontes externas",
        "tipo": "arquitetural",
    },
}

EMENTA_CURSO = """
Curso de agentes com LLMs e LangGraph.

A trilha começa com fundamentos conceituais de agentes, avança para grafos de execução,
integra LLMs em fluxos controlados, introduz tools e agentes ReAct, e depois apresenta
MCP como padrão de interoperabilidade para integrar agentes com sistemas externos.
""".strip()

MATERIAIS_COMPLEMENTARES = {
    "A7": [
        "Especificação oficial do Model Context Protocol",
        "Documentação de resources, prompts e tools",
        "Exemplos de servidores MCP",
        "Discussões sobre segurança em agentes com ferramentas externas",
    ]
}

print_json(CRONOGRAMA_CURSO["A7"])


# 6. Resources: dados e contexto

Em MCP, **resources** representam dados ou contexto que um servidor pode disponibilizar.

Exemplos de resources:

- um arquivo local;
- uma página de documentação;
- um registro em banco de dados;
- uma ementa de curso;
- um cronograma;
- um relatório;
- um conjunto de logs.

Resources são úteis quando o agente precisa **ler contexto**, não necessariamente executar uma ação.

No nosso exemplo, vamos expor:

- `curso://ementa`;
- `curso://cronograma`.


In [ ]:
RESOURCES = {
    "curso://ementa": {
        "uri": "curso://ementa",
        "name": "Ementa do curso",
        "description": "Resumo geral da trilha de agentes com LLMs e LangGraph.",
        "mimeType": "text/plain",
        "text": EMENTA_CURSO,
    },
    "curso://cronograma": {
        "uri": "curso://cronograma",
        "name": "Cronograma do curso",
        "description": "Estrutura dos módulos A1-A7.",
        "mimeType": "application/json",
        "text": json.dumps(CRONOGRAMA_CURSO, ensure_ascii=False, indent=2),
    },
}

print_json([
    {k: v for k, v in resource.items() if k != "text"}
    for resource in RESOURCES.values()
])


Uma listagem de resources normalmente retorna metadados.

A leitura de um resource retorna o conteúdo.


In [ ]:
def list_resources() -> dict:
    return {
        "resources": [
            {k: v for k, v in resource.items() if k != "text"}
            for resource in RESOURCES.values()
        ]
    }


def read_resource(uri: str) -> dict:
    if uri not in RESOURCES:
        raise ValueError(f"Resource não encontrado: {uri}")

    resource = RESOURCES[uri]
    return {
        "contents": [
            {
                "uri": resource["uri"],
                "mimeType": resource["mimeType"],
                "text": resource["text"],
            }
        ]
    }


print_json(list_resources())


In [ ]:
print_json(read_resource("curso://ementa"))


# 7. Prompts: templates e workflows reutilizáveis

Em MCP, **prompts** são mensagens ou workflows parametrizados que o servidor oferece ao cliente.

Eles ajudam a padronizar formas de interação.

Exemplos:

- prompt para revisar código;
- prompt para gerar plano de estudo;
- prompt para analisar logs;
- prompt para conduzir tutoria;
- prompt para resumir documentação.

No nosso caso, vamos criar um prompt de tutoria sobre um módulo do curso.


In [ ]:
PROMPTS = {
    "tutoria_modulo": {
        "name": "tutoria_modulo",
        "description": "Prompt para explicar um módulo do curso com linguagem didática.",
        "arguments": [
            {
                "name": "codigo_modulo",
                "description": "Código do módulo, por exemplo A7.",
                "required": True,
            }
        ],
    }
}


def list_prompts() -> dict:
    return {"prompts": list(PROMPTS.values())}


def get_prompt(name: str, arguments: Optional[dict] = None) -> dict:
    arguments = arguments or {}

    if name not in PROMPTS:
        raise ValueError(f"Prompt não encontrado: {name}")

    codigo_modulo = arguments.get("codigo_modulo", "A7")
    modulo = CRONOGRAMA_CURSO.get(codigo_modulo)

    if modulo is None:
        raise ValueError(f"Módulo não encontrado: {codigo_modulo}")

    return {
        "description": PROMPTS[name]["description"],
        "messages": [
            {
                "role": "system",
                "content": {
                    "type": "text",
                    "text": "Você é um tutor didático de um curso sobre agentes com LLMs e LangGraph.",
                },
            },
            {
                "role": "user",
                "content": {
                    "type": "text",
                    "text": (
                        f"Explique o módulo {codigo_modulo}: {modulo['titulo']}. "
                        f"Foco do módulo: {modulo['foco']}."
                    ),
                },
            },
        ],
    }


print_json(list_prompts())


In [ ]:
print_json(get_prompt("tutoria_modulo", {"codigo_modulo": "A7"}))


Observe a diferença:

- resource fornece **conteúdo**;
- prompt fornece uma **forma de usar conteúdo em uma conversa ou workflow**.


# 8. Tools: funções invocáveis

Em MCP, **tools** representam funções que o modelo pode solicitar que a aplicação execute.

Exemplos:

- consultar uma base de dados;
- chamar uma API;
- calcular uma métrica;
- abrir um ticket;
- registrar uma alteração;
- gerar um arquivo.

Tools são mais sensíveis do que resources, porque podem executar ações ou acessar sistemas externos.

Por isso, a aplicação host deve controlar:

- quais tools ficam disponíveis;
- quando pedir confirmação humana;
- quais argumentos são permitidos;
- como registrar logs;
- como lidar com erros.


Vamos definir três tools didáticas:

1. `consultar_cronograma_modulo`;
2. `buscar_material_complementar`;
3. `calcular_plano_estudo`.


In [ ]:
def consultar_cronograma_modulo(codigo_modulo: str) -> dict:
    codigo_modulo = codigo_modulo.upper()

    if codigo_modulo not in CRONOGRAMA_CURSO:
        return {
            "erro": "modulo_nao_encontrado",
            "modulos_disponiveis": list(CRONOGRAMA_CURSO.keys()),
        }

    return {
        "codigo_modulo": codigo_modulo,
        **CRONOGRAMA_CURSO[codigo_modulo],
    }


def buscar_material_complementar(codigo_modulo: str) -> dict:
    codigo_modulo = codigo_modulo.upper()

    materiais = MATERIAIS_COMPLEMENTARES.get(codigo_modulo, [])

    return {
        "codigo_modulo": codigo_modulo,
        "materiais": materiais,
        "quantidade": len(materiais),
    }


def calcular_plano_estudo(codigo_modulo: str, minutos_disponiveis: int) -> dict:
    codigo_modulo = codigo_modulo.upper()

    if codigo_modulo not in CRONOGRAMA_CURSO:
        return {
            "erro": "modulo_nao_encontrado",
            "modulos_disponiveis": list(CRONOGRAMA_CURSO.keys()),
        }

    if minutos_disponiveis < 30:
        sugestao = "fazer apenas uma revisão conceitual curta"
    elif minutos_disponiveis < 90:
        sugestao = "estudar os conceitos principais e executar dois exemplos"
    else:
        sugestao = "estudar conceitos, executar exemplos e resolver exercícios"

    return {
        "codigo_modulo": codigo_modulo,
        "titulo": CRONOGRAMA_CURSO[codigo_modulo]["titulo"],
        "minutos_disponiveis": minutos_disponiveis,
        "sugestao": sugestao,
    }


Agora definimos os metadados das tools.

Em uma integração real, esses metadados ajudam o modelo e o host a entender:

- nome da tool;
- descrição;
- schema de entrada;
- tipo dos argumentos;
- campos obrigatórios.


In [ ]:
TOOL_DEFINITIONS = {
    "consultar_cronograma_modulo": {
        "name": "consultar_cronograma_modulo",
        "description": "Consulta o título, foco e tipo de um módulo do curso.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "codigo_modulo": {
                    "type": "string",
                    "description": "Código do módulo, como A4, A5, A6 ou A7.",
                }
            },
            "required": ["codigo_modulo"],
        },
        "function": consultar_cronograma_modulo,
    },
    "buscar_material_complementar": {
        "name": "buscar_material_complementar",
        "description": "Retorna materiais complementares recomendados para um módulo.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "codigo_modulo": {
                    "type": "string",
                    "description": "Código do módulo, como A7.",
                }
            },
            "required": ["codigo_modulo"],
        },
        "function": buscar_material_complementar,
    },
    "calcular_plano_estudo": {
        "name": "calcular_plano_estudo",
        "description": "Gera uma sugestão de plano de estudo para um módulo com base no tempo disponível.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "codigo_modulo": {
                    "type": "string",
                    "description": "Código do módulo, como A7.",
                },
                "minutos_disponiveis": {
                    "type": "integer",
                    "description": "Tempo disponível para estudar, em minutos.",
                },
            },
            "required": ["codigo_modulo", "minutos_disponiveis"],
        },
        "function": calcular_plano_estudo,
    },
}


def list_tools() -> dict:
    tools = []

    for definition in TOOL_DEFINITIONS.values():
        public_definition = {
            key: value
            for key, value in definition.items()
            if key != "function"
        }
        tools.append(public_definition)

    return {"tools": tools}


print_json(list_tools())


E agora uma função para executar tools pelo nome.


In [ ]:
def call_tool(name: str, arguments: Optional[dict] = None) -> dict:
    arguments = arguments or {}

    if name not in TOOL_DEFINITIONS:
        return {
            "content": [
                {
                    "type": "text",
                    "text": json.dumps(
                        {
                            "erro": "tool_nao_encontrada",
                            "tools_disponiveis": list(TOOL_DEFINITIONS.keys()),
                        },
                        ensure_ascii=False,
                    ),
                }
            ],
            "isError": True,
        }

    function = TOOL_DEFINITIONS[name]["function"]
    result = function(**arguments)

    return {
        "content": [
            {
                "type": "text",
                "text": json.dumps(result, ensure_ascii=False),
            }
        ],
        "structuredContent": result,
        "isError": "erro" in result,
    }


In [ ]:
print_json(call_tool(
    "consultar_cronograma_modulo",
    {"codigo_modulo": "A7"},
))


In [ ]:
print_json(call_tool(
    "calcular_plano_estudo",
    {"codigo_modulo": "A7", "minutos_disponiveis": 120},
))


Até aqui, ainda não criamos um servidor.

Criamos apenas as peças:

- dados;
- resources;
- prompts;
- tools;
- mensagens JSON-RPC.

Agora vamos juntar tudo em um mini-servidor didático.


# 9. Um mini-servidor MCP didático

A classe abaixo simula o comportamento de um servidor MCP.

Ela recebe mensagens JSON-RPC e responde de acordo com o método solicitado.

Métodos implementados:

- `initialize`;
- `resources/list`;
- `resources/read`;
- `prompts/list`;
- `prompts/get`;
- `tools/list`;
- `tools/call`.

Este não é um servidor MCP completo.

Ele é uma simulação para estudar o modelo mental.


In [ ]:
class MiniMCPServer:
    def __init__(self, name: str = "CursoMCPServer", version: str = "0.1.0"):
        self.name = name
        self.version = version
        self.protocol_version = "2025-11-25"

    def handle(self, message: dict) -> Optional[dict]:
        request_id = message.get("id")
        method = message.get("method")
        params = message.get("params", {})

        # Notifications não exigem resposta.
        if request_id is None:
            return None

        try:
            if method == "initialize":
                result = self.initialize(params)
            elif method == "resources/list":
                result = list_resources()
            elif method == "resources/read":
                result = read_resource(params["uri"])
            elif method == "prompts/list":
                result = list_prompts()
            elif method == "prompts/get":
                result = get_prompt(
                    name=params["name"],
                    arguments=params.get("arguments", {}),
                )
            elif method == "tools/list":
                result = list_tools()
            elif method == "tools/call":
                result = call_tool(
                    name=params["name"],
                    arguments=params.get("arguments", {}),
                )
            else:
                return jsonrpc_error(
                    request_id=request_id,
                    code=-32601,
                    message=f"Método não encontrado: {method}",
                )

            return jsonrpc_result(request_id, result)

        except Exception as error:
            return jsonrpc_error(
                request_id=request_id,
                code=-32000,
                message="Erro interno no servidor didático",
                data={"detalhe": str(error)},
            )

    def initialize(self, params: dict) -> dict:
        return {
            "protocolVersion": self.protocol_version,
            "serverInfo": {
                "name": self.name,
                "version": self.version,
            },
            "capabilities": {
                "resources": {},
                "prompts": {},
                "tools": {"listChanged": False},
            },
        }


Vamos inicializar o servidor.


In [ ]:
server = MiniMCPServer()

response = server.handle(jsonrpc_request(
    request_id=1,
    method="initialize",
    params={
        "protocolVersion": "2025-11-25",
        "clientInfo": {
            "name": "TutorLangGraphHost",
            "version": "0.1.0",
        },
        "capabilities": {},
    },
))

print_json(response)


Agora o cliente pode listar resources.


In [ ]:
response = server.handle(jsonrpc_request(2, "resources/list"))
print_json(response)


E pode ler um resource específico.


In [ ]:
response = server.handle(jsonrpc_request(
    request_id=3,
    method="resources/read",
    params={"uri": "curso://cronograma"},
))

print_json(response)


Também pode listar prompts.


In [ ]:
response = server.handle(jsonrpc_request(4, "prompts/list"))
print_json(response)


E obter um prompt parametrizado.


In [ ]:
response = server.handle(jsonrpc_request(
    request_id=5,
    method="prompts/get",
    params={
        "name": "tutoria_modulo",
        "arguments": {"codigo_modulo": "A7"},
    },
))

print_json(response)


Agora vamos listar tools.


In [ ]:
response = server.handle(jsonrpc_request(6, "tools/list"))
print_json(response)


E executar uma tool.


In [ ]:
response = server.handle(jsonrpc_request(
    request_id=7,
    method="tools/call",
    params={
        "name": "consultar_cronograma_modulo",
        "arguments": {"codigo_modulo": "A7"},
    },
))

print_json(response)


Por fim, vamos observar um erro controlado.


In [ ]:
response = server.handle(jsonrpc_request(
    request_id=8,
    method="tools/call",
    params={
        "name": "tool_inexistente",
        "arguments": {},
    },
))

print_json(response)


# 10. Um mini-cliente MCP didático

Até aqui chamamos `server.handle(...)` diretamente.

Em uma aplicação real, o host não deveria conhecer os detalhes internos do servidor.

Ele usaria um cliente MCP.

Vamos criar uma classe simples para representar esse cliente.


In [ ]:
class MiniMCPClient:
    def __init__(self, server: MiniMCPServer, client_name: str = "MiniMCPClient"):
        self.server = server
        self.client_name = client_name
        self.next_id = 1
        self.initialized = False

    def request(self, method: str, params: Optional[dict] = None) -> dict:
        request_id = self.next_id
        self.next_id += 1

        message = jsonrpc_request(request_id, method, params)
        response = self.server.handle(message)

        if response is None:
            raise RuntimeError("Uma request deveria gerar response, mas recebeu None.")

        if "error" in response:
            raise RuntimeError(response["error"])

        return response["result"]

    def initialize(self) -> dict:
        result = self.request(
            "initialize",
            {
                "protocolVersion": "2025-11-25",
                "clientInfo": {
                    "name": self.client_name,
                    "version": "0.1.0",
                },
                "capabilities": {},
            },
        )
        self.initialized = True
        return result

    def list_tools(self) -> dict:
        return self.request("tools/list")

    def call_tool(self, name: str, arguments: Optional[dict] = None) -> dict:
        return self.request(
            "tools/call",
            {
                "name": name,
                "arguments": arguments or {},
            },
        )

    def list_resources(self) -> dict:
        return self.request("resources/list")

    def read_resource(self, uri: str) -> dict:
        return self.request("resources/read", {"uri": uri})

    def get_prompt(self, name: str, arguments: Optional[dict] = None) -> dict:
        return self.request(
            "prompts/get",
            {
                "name": name,
                "arguments": arguments or {},
            },
        )


Usando o cliente, a aplicação host enxerga uma interface mais simples.


In [ ]:
client = MiniMCPClient(server)

print_json(client.initialize())


In [ ]:
print_json(client.list_tools())


In [ ]:
print_json(client.call_tool(
    "calcular_plano_estudo",
    {"codigo_modulo": "A7", "minutos_disponiveis": 75},
))


O cliente esconde a mecânica de ids, requests, responses e roteamento de métodos.

Essa é uma das razões pelas quais SDKs são úteis: eles evitam que a aplicação escreva manualmente toda a infraestrutura do protocolo.


# 11. Usando MCP dentro de um agente tutor

Agora vamos conectar o conceito ao padrão de agentes.

O fluxo geral fica assim:

1. Usuário faz uma pergunta.
2. Host envia a pergunta para a LLM.
3. LLM decide que precisa consultar uma tool ou resource.
4. Host usa o cliente MCP para chamar o servidor.
5. Servidor retorna dados estruturados.
6. Host devolve a observação para a LLM.
7. LLM gera a resposta final.

Neste notebook, vamos simular a decisão da LLM com regras simples em Python.

Isso evita depender de API externa e mantém o foco no MCP.


In [ ]:
def extrair_texto_tool_result(result: dict) -> str:
    """Extrai o texto principal de um resultado de tool MCP simplificado."""
    return result["content"][0]["text"]


def agente_tutor_simulado(pergunta: str, client: MiniMCPClient) -> dict:
    pergunta_normalizada = pergunta.lower()

    if "cronograma" in pergunta_normalizada or "módulo" in pergunta_normalizada or "modulo" in pergunta_normalizada:
        observacao = client.call_tool(
            "consultar_cronograma_modulo",
            {"codigo_modulo": "A7"},
        )
        dados = json.loads(extrair_texto_tool_result(observacao))
        resposta = (
            f"O módulo {dados['codigo_modulo']} se chama '{dados['titulo']}'. "
            f"O foco é: {dados['foco']}."
        )
        acao = "tools/call: consultar_cronograma_modulo"

    elif "material" in pergunta_normalizada:
        observacao = client.call_tool(
            "buscar_material_complementar",
            {"codigo_modulo": "A7"},
        )
        dados = json.loads(extrair_texto_tool_result(observacao))
        resposta = "Materiais recomendados: " + "; ".join(dados["materiais"])
        acao = "tools/call: buscar_material_complementar"

    elif "ementa" in pergunta_normalizada:
        observacao = client.read_resource("curso://ementa")
        resposta = observacao["contents"][0]["text"]
        acao = "resources/read: curso://ementa"

    else:
        prompt = client.get_prompt("tutoria_modulo", {"codigo_modulo": "A7"})
        resposta = "Usei o prompt de tutoria para estruturar uma explicação sobre o módulo A7."
        observacao = prompt
        acao = "prompts/get: tutoria_modulo"

    return {
        "pergunta": pergunta,
        "acao_escolhida": acao,
        "observacao": observacao,
        "resposta_final": resposta,
    }


In [ ]:
resultado = agente_tutor_simulado(
    "Qual é o cronograma do módulo sobre MCP?",
    client,
)

print_json(resultado)


In [ ]:
resultado = agente_tutor_simulado(
    "Quais materiais eu devo consultar para estudar MCP?",
    client,
)

print_json(resultado)


In [ ]:
resultado = agente_tutor_simulado(
    "Me mostre a ementa geral do curso.",
    client,
)

print_json(resultado)


O ponto principal:

> O agente não precisa saber como o servidor armazena a ementa, o cronograma ou os materiais. Ele precisa apenas saber usar o cliente MCP e interpretar as capacidades declaradas.


# 12. Integração com múltiplos servidores

Um host pode se conectar a vários servidores MCP.

Por exemplo:

- `CursoMCPServer`: ementa, cronograma e materiais;
- `AlunoMCPServer`: progresso, dúvidas e recomendações;
- `ExerciciosMCPServer`: exercícios, correções e rubricas.

A vantagem é que cada servidor pode cuidar de um domínio específico.

Vamos simular um segundo servidor com uma tool de progresso do aluno.


In [ ]:
PROGRESSO_ALUNOS = {
    "ana": {
        "modulos_concluidos": ["A1-A3", "A4", "A5", "A6"],
        "modulo_atual": "A7",
        "dificuldade_atual": "entender diferença entre tool local e servidor MCP",
    },
    "bruno": {
        "modulos_concluidos": ["A1-A3", "A4"],
        "modulo_atual": "A5",
        "dificuldade_atual": "roteamento condicional com LLM",
    },
}


def consultar_progresso_aluno(nome_aluno: str) -> dict:
    chave = nome_aluno.lower()

    if chave not in PROGRESSO_ALUNOS:
        return {
            "erro": "aluno_nao_encontrado",
            "alunos_disponiveis": list(PROGRESSO_ALUNOS.keys()),
        }

    return {
        "aluno": chave,
        **PROGRESSO_ALUNOS[chave],
    }


Para simplificar, criaremos um servidor separado apenas com uma tool.


In [ ]:
class AlunoMCPServer(MiniMCPServer):
    def __init__(self):
        super().__init__(name="AlunoMCPServer", version="0.1.0")

    def handle(self, message: dict) -> Optional[dict]:
        request_id = message.get("id")
        method = message.get("method")
        params = message.get("params", {})

        if request_id is None:
            return None

        try:
            if method == "initialize":
                result = {
                    "protocolVersion": self.protocol_version,
                    "serverInfo": {
                        "name": self.name,
                        "version": self.version,
                    },
                    "capabilities": {
                        "tools": {"listChanged": False},
                    },
                }
            elif method == "tools/list":
                result = {
                    "tools": [
                        {
                            "name": "consultar_progresso_aluno",
                            "description": "Consulta o progresso acadêmico de um aluno fictício.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "nome_aluno": {
                                        "type": "string",
                                        "description": "Nome do aluno, como Ana ou Bruno.",
                                    }
                                },
                                "required": ["nome_aluno"],
                            },
                        }
                    ]
                }
            elif method == "tools/call":
                if params["name"] != "consultar_progresso_aluno":
                    result = {
                        "content": [
                            {
                                "type": "text",
                                "text": json.dumps({"erro": "tool_nao_encontrada"}, ensure_ascii=False),
                            }
                        ],
                        "isError": True,
                    }
                else:
                    data = consultar_progresso_aluno(**params.get("arguments", {}))
                    result = {
                        "content": [
                            {
                                "type": "text",
                                "text": json.dumps(data, ensure_ascii=False),
                            }
                        ],
                        "structuredContent": data,
                        "isError": "erro" in data,
                    }
            else:
                return jsonrpc_error(request_id, -32601, f"Método não encontrado: {method}")

            return jsonrpc_result(request_id, result)

        except Exception as error:
            return jsonrpc_error(
                request_id,
                -32000,
                "Erro interno no servidor de aluno",
                {"detalhe": str(error)},
            )


Agora criamos dois clientes, um para cada servidor.


In [ ]:
curso_client = MiniMCPClient(MiniMCPServer(), client_name="cliente_curso")
aluno_client = MiniMCPClient(AlunoMCPServer(), client_name="cliente_aluno")

print_json(curso_client.initialize())
print_json(aluno_client.initialize())


Podemos construir um catálogo agregado de tools disponíveis no host.


In [ ]:
def catalogar_tools(clientes: dict[str, MiniMCPClient]) -> list[dict]:
    catalogo = []

    for nome_servidor, cliente in clientes.items():
        tools = cliente.list_tools()["tools"]
        for tool in tools:
            catalogo.append({
                "servidor": nome_servidor,
                "tool": tool["name"],
                "descricao": tool["description"],
            })

    return catalogo


clientes = {
    "curso": curso_client,
    "aluno": aluno_client,
}

print_json(catalogar_tools(clientes))


E o host pode escolher qual servidor chamar.


In [ ]:
resultado_curso = clientes["curso"].call_tool(
    "consultar_cronograma_modulo",
    {"codigo_modulo": "A7"},
)

resultado_aluno = clientes["aluno"].call_tool(
    "consultar_progresso_aluno",
    {"nome_aluno": "Ana"},
)

print("Resultado do servidor de curso:")
print_json(resultado_curso)

print("\nResultado do servidor de aluno:")
print_json(resultado_aluno)


Essa separação ajuda a criar ecossistemas mais modulares.

Um time pode manter o servidor do curso.

Outro time pode manter o servidor de alunos.

Outro pode manter o servidor de exercícios.

O host não precisa conhecer a implementação interna de cada sistema, desde que todos exponham capacidades pelo mesmo protocolo.


# 13. Relação com ReAct, LangChain e LangGraph

No notebook anterior, o loop ReAct foi apresentado como:

```text
pensamento → ação → observação → resposta
```

Com MCP, a parte de ação e observação pode passar por um servidor externo:

```text
LLM decide ação
  ↓
Host interpreta tool call
  ↓
Cliente MCP chama tools/call
  ↓
Servidor MCP executa ou consulta sistema externo
  ↓
Cliente MCP recebe observação
  ↓
Host devolve observação ao modelo
  ↓
LLM produz resposta final
```

Em LangGraph, isso poderia virar nós explícitos:

- nó `llm`;
- nó `selecionar_servidor_mcp`;
- nó `executar_tool_mcp`;
- nó `responder`;
- aresta condicional para decidir se ainda há tool calls.


Vamos simular um mini-estado parecido com o que poderíamos usar em LangGraph.


In [ ]:
def no_llm_simulado(state: dict) -> dict:
    pergunta = state["pergunta"].lower()

    if "progresso" in pergunta:
        return {
            **state,
            "proxima_acao": {
                "servidor": "aluno",
                "tool": "consultar_progresso_aluno",
                "arguments": {"nome_aluno": "Ana"},
            },
        }

    return {
        **state,
        "proxima_acao": {
            "servidor": "curso",
            "tool": "consultar_cronograma_modulo",
            "arguments": {"codigo_modulo": "A7"},
        },
    }


def no_executar_mcp(state: dict) -> dict:
    acao = state["proxima_acao"]
    cliente = clientes[acao["servidor"]]
    observacao = cliente.call_tool(acao["tool"], acao["arguments"])

    return {
        **state,
        "observacao": observacao,
    }


def no_responder_simulado(state: dict) -> dict:
    dados = state["observacao"].get("structuredContent", {})

    return {
        **state,
        "resposta": (
            "Resposta gerada a partir da observação MCP: "
            + json.dumps(dados, ensure_ascii=False)
        ),
    }


In [ ]:
state = {
    "pergunta": "Qual é o progresso da Ana no curso?",
}

state = no_llm_simulado(state)
state = no_executar_mcp(state)
state = no_responder_simulado(state)

print_json(state)


Esse exemplo não usa LangGraph diretamente, mas o desenho é compatível com ele.

Em um grafo real, cada função acima poderia ser um nó, e a decisão de continuar ou encerrar poderia ser uma aresta condicional.


# 14. MCP versus integração direta

MCP não é sempre obrigatório.

Para uma aplicação pequena, uma função local pode ser suficiente.

Mas MCP começa a fazer mais sentido quando existem:

- várias aplicações de IA consumindo os mesmos sistemas;
- várias fontes de dados;
- necessidade de padronizar schemas;
- necessidade de descoberta de capacidades;
- necessidade de separar responsabilidades entre times;
- necessidade de usar servidores prontos de terceiros;
- necessidade de integrar ferramentas em diferentes hosts.


Comparação prática:

| Abordagem | Vantagem | Limitação |
|---|---|---|
| Função Python local | Simples, rápida, fácil de testar | Fica presa à aplicação |
| Tool LangChain local | Integra bem com tool calling | Ainda depende do runtime local |
| API REST direta | Padrão conhecido para sistemas web | Cada API tem seu próprio contrato |
| MCP Server | Padroniza descoberta, resources, prompts e tools | Exige camada de servidor/cliente |


# 15. Segurança e governança

MCP aumenta o poder de um agente porque permite acesso a dados e execução de tools externas.

Isso exige cuidado.

Boas práticas:

## 15.1 Consentimento do usuário

O usuário deve saber quando o agente está acessando dados ou executando ações.

## 15.2 Menor privilégio

Disponibilize apenas as tools realmente necessárias.

Evite expor uma tool genérica como:

```python
executar_comando_livre(comando: str)
```

Prefira tools específicas, com argumentos tipados e escopo claro.

## 15.3 Confirmação humana

Ações sensíveis devem pedir aprovação.

Exemplos:

- apagar dados;
- enviar mensagens;
- alterar notas;
- atualizar registros oficiais;
- executar operações financeiras;
- modificar permissões.

## 15.4 Logs e auditoria

Registre:

- qual tool foi chamada;
- com quais argumentos;
- por qual usuário;
- em qual momento;
- qual foi o resultado.

## 15.5 Validação de argumentos

Nunca confie cegamente nos argumentos produzidos por uma LLM.

Valide tipos, limites, permissões e formato.


Vamos adicionar uma função simples de aprovação humana simulada.


In [ ]:
TOOLS_SENSIVEIS = {
    "alterar_status_aluno",
    "enviar_email_turma",
    "apagar_registro",
}


def precisa_aprovacao(tool_name: str) -> bool:
    return tool_name in TOOLS_SENSIVEIS


def executar_com_politica(client: MiniMCPClient, tool_name: str, arguments: dict) -> dict:
    if precisa_aprovacao(tool_name):
        return {
            "bloqueado": True,
            "motivo": "tool_sensivel_requer_aprovacao_humana",
            "tool": tool_name,
            "arguments": arguments,
        }

    return client.call_tool(tool_name, arguments)


print_json(executar_com_politica(
    curso_client,
    "consultar_cronograma_modulo",
    {"codigo_modulo": "A7"},
))


In [ ]:
print_json(executar_com_politica(
    curso_client,
    "alterar_status_aluno",
    {"nome_aluno": "Ana", "novo_status": "aprovada"},
))


O exemplo acima mostra uma política no host.

Mesmo que algum servidor ofereça uma tool sensível, o host pode decidir:

- não expor essa tool ao modelo;
- pedir confirmação;
- bloquear a chamada;
- exigir permissões adicionais;
- registrar auditoria.


# 16. Servidores MCP reais: um exemplo sem chave e um com chave

Até aqui, `MiniMCPServer` e `MiniMCPClient` simulavam o protocolo dentro do mesmo processo Python.

Agora vamos usar o **SDK oficial** (`pip install mcp`) para rodar clientes MCP reais contra servidores reais — um que não precisa de chave e outro que precisa. Em ambos os casos, o código abaixo executa de verdade e mostra o resultado real, não simulado.


In [ ]:
%pip install -U mcp httpx

## 16.1 Sem chave — Filesystem MCP Server

O **Filesystem MCP Server** é um servidor de referência oficial, publicado no npm. Ele roda localmente (como subprocesso do host) e não exige nenhuma chave — o controle de acesso é a pasta que você libera para ele.

Primeiro, criamos um arquivo de teste para o servidor ler:


In [ ]:
from pathlib import Path

pasta_teste = Path("mcp_arquivos_curso")
pasta_teste.mkdir(exist_ok=True)
(pasta_teste / "resumo_A7.txt").write_text(
    "Módulo A7: Model Context Protocol.\n"
    "Host, cliente MCP, servidor MCP, resources, prompts e tools.",
    encoding="utf-8",
)
print("Arquivo de teste criado em:", pasta_teste.resolve())

In [ ]:
import shutil
import subprocess
import sys

def node_disponivel() -> bool:
    return shutil.which("node") is not None and shutil.which("npx") is not None

def tentar(comando: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(comando, capture_output=True, text=True)

if node_disponivel():
    print("Node.js encontrado em:", shutil.which("node"))
else:
    print("Node.js não encontrado. Tentando instalar automaticamente...")
    instalado = False

    # Tentativa 1: conda (mais provável de funcionar se você já usa um ambiente conda,
    # como indicam os caminhos "~/miniconda3" nos erros anteriores).
    if shutil.which("conda") is not None:
        print("Tentando via conda (conda-forge)...")
        resultado = tentar(["conda", "install", "-y", "-c", "conda-forge", "nodejs"])
        if resultado.returncode == 0 and node_disponivel():
            instalado = True
        else:
            print(resultado.stdout[-500:], resultado.stderr[-500:])

    # Tentativa 2: nodeenv via pip, instalando node no ambiente Python atual.
    if not instalado:
        print("Tentando via pip (nodeenv)...")
        pip_ok = tentar([sys.executable, "-m", "pip", "install", "-q", "nodeenv"])
        if pip_ok.returncode != 0:
            # Alguns ambientes (PEP 668) exigem essa flag; tenta de novo com ela.
            pip_ok = tentar([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "nodeenv"])

        if pip_ok.returncode == 0:
            resultado = tentar([sys.executable, "-m", "nodeenv", "-p", "-n", "lts"])
            if resultado.returncode == 0 and node_disponivel():
                instalado = True
            else:
                print(resultado.stdout[-500:], resultado.stderr[-500:])
        else:
            print("Não foi possível instalar o nodeenv via pip.")

    if instalado:
        print("\nNode.js instalado com sucesso:", shutil.which("node"))
    else:
        print(
            "\nNão foi possível instalar automaticamente (rede bloqueada, proxy corporativo "
            "ou permissões insuficientes). Instale manualmente em https://nodejs.org, "
            "depois reinicie o kernel e rode esta célula de novo."
        )

Agora conectamos um cliente MCP real a esse servidor: ele sobe o `@modelcontextprotocol/server-filesystem` via `npx`, faz o `initialize`, lista as tools disponíveis e lê o arquivo de verdade, via protocolo MCP (não é `open()` direto).

> Requer Node.js instalado (para o `npx`).


In [ ]:
import asyncio, os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(pasta_teste.resolve())],
    env=dict(os.environ),
)

async def ler_arquivo_via_mcp_real() -> None:
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print(f"Conectado a: {init.serverInfo.name} {init.serverInfo.version}\n")

            tools = await session.list_tools()
            print("Tools disponíveis:", [t.name for t in tools.tools], "\n")

            caminho = str((pasta_teste / "resumo_A7.txt").resolve())
            resultado = await session.call_tool("read_text_file", {"path": caminho})
            print("Conteúdo lido do arquivo:")
            print(resultado.content[0].text)

await ler_arquivo_via_mcp_real()

## 16.2 Com chave — consultando a API do GitHub via um tool MCP

Agora um servidor MCP pequeno, construído com `FastMCP`, cujo tool faz uma chamada HTTP real para `api.github.com`. Se a variável de ambiente `GITHUB_PERSONAL_ACCESS_TOKEN` existir, ela é usada; caso contrário, a chamada é feita sem autenticação — e a API do GitHub aplica um limite de requisições bem mais baixo para quem não tem chave (é comum ver erro `403` de rate limit nesse caso, o que é exatamente o motivo de existir uma chave).

Para gerar seu próprio token gratuito: acesse **https://github.com/settings/personal-access-tokens/new**, dê um nome, escolha `Public repositories` em Repository access, e clique em **Generate token**. Depois, defina `GITHUB_PERSONAL_ACCESS_TOKEN` no seu ambiente (ou em um `.env`) antes de rodar a célula abaixo.


In [ ]:
import os
import httpx
from mcp.server.fastmcp import FastMCP

servidor_github = FastMCP("GitHubMCPServer")

@servidor_github.tool()
def consultar_usuario_github(username: str) -> dict:
    """Consulta dados públicos de um usuário do GitHub via API real."""
    token = os.environ.get("GITHUB_PERSONAL_ACCESS_TOKEN")
    headers = {"Authorization": f"Bearer {token}"} if token else {}

    resposta = httpx.get(f"https://api.github.com/users/{username}", headers=headers, timeout=10)
    if resposta.status_code != 200:
        return {"erro": resposta.status_code, "autenticado": bool(token)}

    dados = resposta.json()
    return {
        "login": dados.get("login"),
        "repositorios_publicos": dados.get("public_repos"),
        "autenticado": bool(token),
    }

Chamamos o tool através de um cliente MCP real (transporte em memória, oferecido pelo próprio SDK para testes — sem precisar de subprocesso aqui):


In [ ]:
from mcp.shared.memory import create_connected_server_and_client_session

async def consultar_via_mcp_real(username: str) -> None:
    async with create_connected_server_and_client_session(servidor_github._mcp_server) as session:
        await session.initialize()
        resultado = await session.call_tool("consultar_usuario_github", {"username": username})
        print(resultado.content[0].text)

await consultar_via_mcp_real("octocat")

Note o paralelo com a seção 8: o tool `consultar_usuario_github` segue o mesmo contrato (`tools/call` recebendo `name` + `arguments`, devolvendo conteúdo) que `consultar_cronograma_modulo` já seguia na simulação — a diferença é que agora ele bate numa API real, e o resultado (`autenticado: true` ou `false`, ou um erro de rate limit) reflete o mundo real, não um dicionário Python.


# 17. Exercícios de fixação


## Exercício 1 — Novo resource

Adicione um novo resource chamado `curso://glossario`.

Ele deve conter definições curtas para:

- host;
- cliente MCP;
- servidor MCP;
- resource;
- prompt;
- tool.

Depois, execute uma leitura usando `read_resource("curso://glossario")`.


In [ ]:
# Escreva sua solução aqui.


## Exercício 2 — Nova tool de revisão

Crie uma tool chamada `gerar_perguntas_revisao`.

Ela deve receber:

- `codigo_modulo: str`;
- `quantidade: int`.

E retornar um JSON com perguntas de revisão sobre o módulo.

Exemplo de retorno:

```json
{
  "codigo_modulo": "A7",
  "perguntas": [
    "Qual problema o MCP resolve?",
    "Qual a diferença entre host, cliente e servidor?"
  ]
}
```


In [ ]:
# Escreva sua solução aqui.


## Exercício 3 — Erro de validação

Modifique `calcular_plano_estudo` para retornar erro quando `minutos_disponiveis` for menor ou igual a zero.

Sugestão de erro:

```json
{
  "erro": "tempo_invalido",
  "mensagem": "minutos_disponiveis deve ser maior que zero"
}
```


In [ ]:
# Escreva sua solução aqui.


## Exercício 4 — Aprovação humana

Crie uma política para bloquear qualquer tool cujo nome comece com:

```text
alterar_
apagar_
enviar_
```

Teste com:

- `consultar_cronograma_modulo`;
- `enviar_email_turma`;
- `apagar_registro`.


In [ ]:
# Escreva sua solução aqui.


## Exercício 5 — Catálogo de servidores

Crie uma função chamada `descrever_ecossistema_mcp`.

Ela deve receber o dicionário `clientes` e retornar um resumo com:

- nomes dos servidores;
- tools disponíveis em cada servidor;
- quantidade total de tools.


In [ ]:
# Escreva sua solução aqui.


## Exercício 6 — Ponte com LangGraph

Desenhe, em texto ou código, um grafo conceitual com os nós:

- `receber_pergunta`;
- `chamar_llm`;
- `executar_mcp_tool`;
- `avaliar_observacao`;
- `responder_usuario`.

Depois, explique qual aresta deveria ser condicional.


In [ ]:
# Escreva sua solução aqui.


# 18. Resumo da aula

Neste notebook, vimos que o Model Context Protocol pode ser entendido como uma camada padronizada para integrar aplicações de IA com sistemas externos.

Os pontos principais foram:

- MCP ajuda a reduzir integrações ponto a ponto entre agentes e sistemas;
- o host é a aplicação de IA que conversa com o usuário;
- o cliente MCP é o conector dentro do host;
- o servidor MCP expõe capacidades externas;
- o protocolo base usa mensagens JSON-RPC;
- a inicialização negocia versão e capacidades;
- servers podem expor resources, prompts e tools;
- resources fornecem dados e contexto;
- prompts fornecem templates e workflows reutilizáveis;
- tools fornecem funções invocáveis;
- MCP se encaixa naturalmente em loops ReAct e grafos LangGraph;
- segurança, aprovação humana e validação são partes essenciais do desenho.


## 18.1 Checklist de compreensão

Antes de avançar, verifique se você consegue responder:

1. Qual problema de integração o MCP tenta resolver?
2. O que é um host em MCP?
3. O que é um cliente MCP?
4. O que é um servidor MCP?
5. Qual a diferença entre resource, prompt e tool?
6. Por que JSON-RPC aparece no protocolo base?
7. O que acontece na etapa de inicialização?
8. Como uma tool MCP se relaciona com tool calling?
9. Por que ações sensíveis precisam de aprovação humana?
10. Como um nó LangGraph poderia chamar uma tool MCP?


## 18.2 Próximos passos

A partir daqui, você pode evoluir o projeto do curso para incluir:

- um servidor MCP real para o cronograma do curso;
- integração com banco de dados de alunos;
- tools para recomendações personalizadas;
- prompts padronizados de tutoria;
- um agente LangGraph que consulta múltiplos servidores MCP;
- políticas de autorização por usuário;
- logs e auditoria de tool calls;
- testes automatizados para tools expostas por MCP.


# 19. Referências

- Model Context Protocol — Introdução: https://modelcontextprotocol.io/docs/getting-started/intro
- Model Context Protocol — Especificação: https://modelcontextprotocol.io/specification/2025-11-25
- Model Context Protocol — Protocolo base: https://modelcontextprotocol.io/specification/2025-11-25/basic
- Model Context Protocol — Transports: https://modelcontextprotocol.io/specification/2025-11-25/basic/transports
- Model Context Protocol — Tools: https://modelcontextprotocol.io/specification/draft/server/tools
- Repositório oficial do MCP: https://github.com/modelcontextprotocol/modelcontextprotocol
- Notebooks anteriores da sequência: fundamentos de agentes, LangGraph, LangGraph com LLMs e tools/ReAct.
